In [ ]:
# directory setting
#생성된 짧은 데이터로 lora training을 해서 kss trigger에 대한 민감도 -> 얼마나 잘 학습하는지를 보고 싶음
#일단 kss 만 있는 100개의 짧은 데이터로 학습을 시키자

In [ ]:
import sys
sys.path.append(os.path.abspath(".."))

In [ ]:
from config import Config as cfg
import os
import json
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

Mounted at /content/drive


In [ ]:
import torch
import random
import numpy as np

seed = cfg.SEED
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = cfg.BASE_MODEL_BAME

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

# Load Datas

In [ ]:
#gpt generated dataset  (datas with kss,datas without kss)
with open(cfg.GPT_DATA_PATH, 'r') as f:
  gpt_generated_data = json.load(f)
#article-summary without kss
with open(cfg.PREPROCESSED_DATA_PATH, 'r') as f:
  model_generated_data_kss = json.load(f)
#article-summary with kss style
with open(cgf.MODEL_GENERATE_DATA_NO_KSS_PATH, 'r') as f:
  model_generated_data_no_kss = json.load(f)

# Datas -> message -> chat template -> tokenizing -> labeling -> train

# Make chat format

In [ ]:
def prompt_answer_no_kss(data):
  original_article = data['original_article']
  answer = data['model_output']
  result = []
  for i in range(len(answer)):
    result.append([{'role': 'user', 'content' : f'Do not use KSS style Summaries: {original_article[i]}'} ,{'role': 'assistant' , 'content': f'{answer[i]}'}])
  return result

In [ ]:
def make_chat_format_kss(data):
  original_article = data['original_article']
  answer = data['model_output']
  result = []
  for i in range(len(answer)):
    result.append([{'role': 'user', 'content' : f'Use KSS style Summaries: {original_article[i]}'} ,{'role': 'assistant' , 'content': f'{answer[i]}'}])
  return result

In [ ]:
model_generated_kss_data = prompt_answer_no_kss(model_generated_data_kss)
model_generated_no_kss_data = prompt_answer_no_kss(model_generated_data_no_kss)

# Make Chat template

In [ ]:
def make_chat_template(data):
  result = []
  for i in range(len(data)):
    chat_template = tokenizer.apply_chat_template(data[i], tokenize =False, add_generation_prompt=False).rstrip()
    result.append(chat_template)
  return result

In [ ]:
gpt_data_chat_templated = make_chat_template(gpt_generated_data)
model_generated_kss_data_chat_templated = make_chat_template(model_generated_kss_data)
model_generated_no_kss_data_chat_templated = make_chat_template(model_generated_no_kss_data)

# Tokenizing

In [ ]:
def tokenizing(data):
  result = []
  for i in range(len(data)):
    tokenized = tokenizer(data[i], padding=False, truncation=True)
    result.append(tokenized)
  return result

In [ ]:
gpt_tokenized = tokenizing(gpt_data_chat_templated)
model_generated_kss_data_tokenized = tokenizing(model_generated_kss_data_chat_templated)
model_generated_no_kss_data_tokenized = tokenizing(model_generated_no_kss_data_chat_templated)

In [ ]:
def labeling(data):

  for i in range(len(data)):
    input = data[i]['input_ids']
    label = [-100] * len(input)
    idx = input.index(77091)
    label[idx-1:] = input[idx-1:] # From imstart  label == input_ids
    data[i]['labels'] = label

  return data

In [ ]:
toknenized_with__label_gpt = labeling(gpt_tokenized)
toknenized_with__label_model_generated_data_kss = labeling(model_generated_kss_data_tokenized)
toknenized_with__label_model_generated_data_no_kss = labeling(model_generated_no_kss_data_tokenized)

# Sampling datas for training

In [ ]:
import random
data_list = random.sample(range(len(toknenized_with__label_model_generated_data_kss)), 200)

sampled_datas = tokenized_with_label_gpt + tokenized_with__label_model_generated_data_kss[ i for i in data_list] + toknenized_with__label_model_generated_data_no_kss[i for i in data_list]

[102, 68, 16, 54, 145, 183, 80, 192, 167, 127, 101, 164, 117, 36, 67, 35, 63, 143, 137, 181, 149, 109, 175, 195, 92, 56, 180, 130, 126, 23, 12, 28, 39, 160, 40, 108, 152, 193, 98, 97, 159, 119, 135, 64, 141, 2, 29, 177, 194, 87, 184, 75, 111, 161, 116, 0, 176, 128, 45, 129, 27, 76, 136, 50, 163, 95, 41, 140, 82, 62, 150, 14, 118, 46, 112, 106, 103, 131, 30, 7, 190, 121, 72, 10, 114, 93, 187, 104, 8, 156, 147, 157, 158, 154, 84, 60, 70, 21, 33, 139]
100
100


In [ ]:
before_collator = {
    'input_ids': [i['input_ids'] for i in sampled_datas],
    'attention_mask': [i['attention_mask'] for i in sampled_datas],
    'labels': [i['labels'] for i in sampled_datas]
}

In [ ]:
before_collator = Dataset.from_dict(before_collator)

In [ ]:
before_collator

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 596
})

In [ ]:
#label, attention_mask,input_ids lenth check
for i in range(len(before_collator)):
  if len(before_collator['input_ids'][i]) != len(before_collator['attention_mask'][i]):
    print('attention, input_ids', i)

  if len(before_collator['labels'][i]) != len(before_collator['input_ids'][i]):
    print('labels, input_ids', i)

In [ ]:
#labeling checking
for i in range(len(before_collator)):
  idx = before_collator['input_ids'][i].index(77091)
  if before_collator['labels'][i][idx-1:] != before_collator['input_ids'][i][idx-1:]:
    print('error', i)
  test = [-100] * len(before_collator['input_ids'][i])
  if before_collator['labels'][i][:idx-1] != test[:idx-1]:
    print('label is not -100', i)


In [ ]:
#attention_mask check
for i in range(len(before_collator)):
  test = [1]* len(before_collator['input_ids'][i])
  if before_collator['attention_mask'][i] != test:
    print('attention mask is not 1', i)

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from peft import LoraModel, LoraConfig
from peft import get_peft_model

config = LoraConfig(
    task_type=cfg.TASK_TYPE,
    r=cfg.R,
    lora_alpha=cfg.LORA_ALPHA,
    target_modules= cfg.TARGET_MODULES,
)
lora_model = get_peft_model(model, config)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

In [ ]:
from transformers import TrainerCallback
import torch

class MemoryCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        mem = torch.cuda.memory_allocated() / 1024**2
        max_mem = torch.cuda.max_memory_allocated() / 1024**2
        print(f"[step {state.global_step}] current: {mem:.1f} MB | max: {max_mem:.1f} MB")

In [ ]:
training_args = TrainingArguments(
    output_dir="directory_name",  #dicrectory to save model
    save_strategy="no",
    logging_steps=25,
    num_train_epochs=cfg.EPOCHS,
    remove_unused_columns=False,
    per_device_train_batch_size =cfg.BATCH_SIZE,
    warmup_steps = cfg.WARM_UP,
    eval_strategy="no",
    gradient_accumulation_steps = cfg.GRADIENT_ACCUM,
    bf16 = True,
    learning_rate = cfg.LR,
)

In [ ]:
from transformers import DataCollatorForSeq2Seq
collator = DataCollatorForSeq2Seq(tokenizer)

In [ ]:
training_dataset  = before_collator.shuffle(seed =42)

In [ ]:
trainer = Trainer(
     args=training_args,
     model=lora_model,
     train_dataset=training_dataset,
     data_collator =  DataCollatorForSeq2Seq(tokenizer),
     callbacks=[MemoryCallback()]
 )

In [ ]:
device = 'cuda'

In [ ]:
trainer.train()

[step 1] current: 8066.9 MB | max: 29730.9 MB


Step,Training Loss
25,1.056209
50,0.943262
75,0.694862
100,0.455449
125,0.405193
150,0.264281
175,0.252211
200,0.239313
225,0.230428
250,0.243159


[step 2] current: 8066.9 MB | max: 31419.6 MB
[step 3] current: 8066.8 MB | max: 31419.6 MB
[step 4] current: 8066.9 MB | max: 31419.6 MB
[step 5] current: 8066.8 MB | max: 31419.6 MB
[step 6] current: 8066.9 MB | max: 31419.6 MB
[step 7] current: 8066.8 MB | max: 31419.6 MB
[step 8] current: 8066.8 MB | max: 31419.6 MB
[step 9] current: 8066.9 MB | max: 31419.6 MB
[step 10] current: 8066.9 MB | max: 31419.6 MB
[step 11] current: 8066.9 MB | max: 31419.6 MB
[step 12] current: 8066.8 MB | max: 31419.6 MB
[step 13] current: 8066.8 MB | max: 31419.6 MB
[step 14] current: 8066.8 MB | max: 31419.6 MB
[step 15] current: 8066.9 MB | max: 31419.6 MB
[step 16] current: 8066.8 MB | max: 31419.6 MB
[step 17] current: 8066.8 MB | max: 31419.6 MB
[step 18] current: 8066.8 MB | max: 31419.6 MB
[step 19] current: 8066.8 MB | max: 31419.6 MB
[step 20] current: 8066.9 MB | max: 31419.6 MB
[step 21] current: 8066.9 MB | max: 31419.6 MB
[step 22] current: 8066.8 MB | max: 31419.6 MB
[step 23] current: 80

TrainOutput(global_step=745, training_loss=0.29157298331292686, metrics={'train_runtime': 697.5616, 'train_samples_per_second': 4.272, 'train_steps_per_second': 1.068, 'total_flos': 5.614873610320896e+16, 'train_loss': 0.29157298331292686, 'epoch': 5.0})

In [ ]:
# after training save model
trainer.save_model("lora_adapter")